In [1]:
!pip install spektral

You should consider upgrading via the '/usr/bin/python3 -m pip install --upgrade pip' command.


In [1]:
import os
import numpy as np
import pandas as pd
from scipy import sparse
import tensorflow as tf
from tensorflow.keras.layers import Dense, Dropout,Input
from tensorflow.keras.losses import CategoricalCrossentropy
from tensorflow.keras.metrics import categorical_accuracy
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import Adam
from spektral.layers import GINConv,GCNConv #, GCSConv, GlobalAvgPool
from spektral.utils.sparse import sp_matrix_to_sp_tensor
from spektral.data import DisjointLoader, BatchLoader, Dataset, Graph
#from spektral.transforms.normalize_adj import NormalizeAdj
import gc
import spektral.datasets

In [15]:
os.environ['SPEKTRAL_DATA'] = "/mnt/grafos_npz2/"

In [7]:
class MyDataset(Dataset):
    """
    A dataset of random colored graphs.
    The task is to classify each graph with the color which occurs the most in
    its nodes.
    The graphs have `n_colors` colors, of at least `n_min` and at most `n_max`
    nodes connected with probability `p`.
    """

    def __init__(self, path, **kwargs):
        super().__init__(**kwargs)
        self.path = path

    def download(self):      
        captures = ["10","11","12","15","15-2","16","16-2","16-3","17","18","18-2","19","15-3"]
        #fn_ncol_int = os.path.join(self.path,"edges_int/edges_INT_capture201108", self.cap, ".ncol")
        #fn_features_int = os.path.join(self.path,"features_int/features_INT_capture201108", self.cap, ".csv")

        # Write the data to file
        for i in captures:
            #x_tmp = pd.read_csv(os.path.join(self.path, f'edges_int/features_INT_capture201108{i}.csv'), sep=",", header=0)
            x_tmp = pd.read_csv(f'/features-prefix/capture201108{i}_features_prefix.csv', sep=",", header=0)
            class_idx = {name: idx for idx, name in enumerate(sorted(x_tmp["label"].unique()))}
            node_idx = {name: idx for idx, name in enumerate(sorted(x_tmp["node"].unique()))}
            # Cambiamos los nodos y clases por su correspondiente número entero, en las features y en los grafos
            x_tmp["node"] = x_tmp["node"].apply(lambda name: node_idx[name])
            x_tmp["label"] = x_tmp["label"].apply(lambda value: class_idx[value])
            feature_names = set(x_tmp.columns) - {"node", "label"}
            x = tf.cast(x_tmp.sort_values("node")[feature_names].to_numpy(), dtype=tf.dtypes.float32)        
            
            #a_temp = pd.read_csv(os.path.join(self.path, f'edges_int/edges_INT_capture201108{i}.ncol'), sep=" ", header=None, names=["source", "target", "weight"])
            a_tmp = pd.read_csv(f'/ncol-prefix/capture201108{i}_ncol_prefix.ncol', sep=" ", header=None, names=["source", "target", "weight"])
            a_tmp["source"] = a_tmp["source"].apply(lambda name: node_idx[name])
            a_tmp["target"] = a_tmp["target"].apply(lambda name: node_idx[name])
            a_source_tmp = a_tmp[["source"]].to_numpy().T
            a_source = np.reshape(a_source_tmp, a_source_tmp.shape[-1])
            a_target_tmp = a_tmp[["target"]].to_numpy().T
            a_target = np.reshape(a_target_tmp, a_target_tmp.shape[-1])
            a_weight_tmp = a_tmp[["weight"]].to_numpy().T
            a_weight = np.reshape(a_weight_tmp, a_weight_tmp.shape[-1])
            a = sparse.coo_matrix((a_weight, (a_source, a_target)), shape=(x.shape[0], x.shape[0]))

            y = tf.cast(x_tmp.sort_values("node")["label"].to_numpy(), dtype=tf.dtypes.int64)

            #filename = f'/grafos_npz/graph_201108{i}.npz'
            filename = os.path.join(self.path, f'/grafos_npz/graph_201108{i}.npz')
            np.savez(filename, x=x, a=a, y=y)
            print(i,"final\n")
            del x_tmp, feature_names, a_tmp, a_source_tmp, a_source, a_target_tmp, a_target, a_weight_tmp, a_weight
            gc.collect()


    def read(self):
       # We must return a list of Graph objects
        output = []
        
        captures = ["10","11","12","15","15-2","16","16-2","16-3","17","18","18-2","19","15-3"]

        for i in captures:
            #data = np.load(f'/grafos_npz/graph_201108{i}.npz')#, allow_pickle=True)
            data = np.load(os.path.join(self.path, f'/mnt/grafos_npz/graph_201108{i}.npz'), allow_pickle=True)
            output.append(
                Graph(x=data['x'], a=data['a'][()], y=data['y'])
            )

        return output


In [14]:
captures = ["10","11","12","15","15-2","16","16-2","16-3","17","18","18-2","19","15-3"]
        #fn_ncol_int = os.path.join(self.path,"edges_int/edges_INT_capture201108", self.cap, ".ncol")
        #fn_features_int = os.path.join(self.path,"features_int/features_INT_capture201108", self.cap, ".csv")

        # Write the data to file
for i in captures:
    #x_tmp = pd.read_csv(os.path.join(self.path, f'edges_int/features_INT_capture201108{i}.csv'), sep=",", header=0)
    x_tmp = pd.read_csv(f'/mnt/features-prefix/capture201108{i}_features_prefix.csv', sep=",", header=0)
    class_idx = {name: idx for idx, name in enumerate(sorted(x_tmp["label"].unique()))}
    node_idx = {name: idx for idx, name in enumerate(sorted(x_tmp["node"].unique()))}
    # Cambiamos los nodos y clases por su correspondiente número entero, en las features y en los grafos
    x_tmp["node"] = x_tmp["node"].apply(lambda name: node_idx[name])
    x_tmp["label"] = x_tmp["label"].apply(lambda value: class_idx[value])
    feature_names = set(x_tmp.columns) - {"node", "label"}
    x = tf.cast(x_tmp.sort_values("node")[feature_names].to_numpy(), dtype=tf.dtypes.float32)        
            
    #a_temp = pd.read_csv(os.path.join(self.path, f'edges_int/edges_INT_capture201108{i}.ncol'), sep=" ", header=None, names=["source", "target", "weight"])
    a_tmp = pd.read_csv(f'/mnt/ncol-prefix/capture201108{i}_ncol_prefix.ncol', sep=" ", header=None, names=["source", "target", "weight"])
    a_tmp["source"] = a_tmp["source"].apply(lambda name: node_idx[name])
    a_tmp["target"] = a_tmp["target"].apply(lambda name: node_idx[name])
    a_source_tmp = a_tmp[["source"]].to_numpy().T
    a_source = np.reshape(a_source_tmp, a_source_tmp.shape[-1])
    a_target_tmp = a_tmp[["target"]].to_numpy().T
    a_target = np.reshape(a_target_tmp, a_target_tmp.shape[-1])
    a_weight_tmp = a_tmp[["weight"]].to_numpy().T
    a_weight = np.reshape(a_weight_tmp, a_weight_tmp.shape[-1])
    a = sparse.coo_matrix((a_weight, (a_source, a_target)), shape=(x.shape[0], x.shape[0]))

    y = tf.cast(x_tmp.sort_values("node")["label"].to_numpy(), dtype=tf.dtypes.int64)

    filename = f'/mnt/grafos_npz/graph_201108{i}.npz'
            #filename = os.path.join(self.path, f'grafos_npz/graph_201108{i}.npz')
    np.savez(filename, x=x, a=a, y=y)
    
    del x_tmp, feature_names, a_tmp, a_source_tmp, a_source, a_target_tmp, a_target, a_weight_tmp, a_weight
    gc.collect()


<ipython-input-14-d1d7eef30db4>:15: FutureWarning: Passing a set as an indexer is deprecated and will raise in a future version. Use a list instead.
  x = tf.cast(x_tmp.sort_values("node")[feature_names].to_numpy(), dtype=tf.dtypes.float32)


10 final



<ipython-input-14-d1d7eef30db4>:15: FutureWarning: Passing a set as an indexer is deprecated and will raise in a future version. Use a list instead.
  x = tf.cast(x_tmp.sort_values("node")[feature_names].to_numpy(), dtype=tf.dtypes.float32)


11 final



<ipython-input-14-d1d7eef30db4>:15: FutureWarning: Passing a set as an indexer is deprecated and will raise in a future version. Use a list instead.
  x = tf.cast(x_tmp.sort_values("node")[feature_names].to_numpy(), dtype=tf.dtypes.float32)


...

18-2 final



<ipython-input-14-d1d7eef30db4>:15: FutureWarning: Passing a set as an indexer is deprecated and will raise in a future version. Use a list instead.
  x = tf.cast(x_tmp.sort_values("node")[feature_names].to_numpy(), dtype=tf.dtypes.float32)


19 final



<ipython-input-14-d1d7eef30db4>:15: FutureWarning: Passing a set as an indexer is deprecated and will raise in a future version. Use a list instead.
  x = tf.cast(x_tmp.sort_values("node")[feature_names].to_numpy(), dtype=tf.dtypes.float32)


15-3 final



In [28]:
class MyDataset(Dataset):
    """
    A dataset of random colored graphs.
    The task is to classify each graph with the color which occurs the most in
    its nodes.
    The graphs have `n_colors` colors, of at least `n_min` and at most `n_max`
    nodes connected with probability `p`.
    """

    #def __init__(self, **kwargs):
    #    #self.path = path
    #    super().__init__(**kwargs)

    def __init__(self, **kwargs):
        super().__init__(**kwargs)
        #self.path = path
        

    def read(self):
       # We must return a list of Graph objects
        output = []
        
        captures = ["10","11","12","15","15-2","16","16-2","18","18-2","15-3"]

        for i in captures:
            #data = np.load(f'/mnt/grafos_npz/graph_201108{i}.npz', allow_pickle=True)
            #data = np.load(os.path.join(self.path, f'graph_201108{i}.npz'))
            data = np.load(f'/mnt/grafos_npz2/graph_201108{i}.npz', allow_pickle=True)
            output.append(
                Graph(x=data['x'], a=data['a'], y=data['y'])
            )

        return output


In [67]:
output = []
captures = ["10","11","12","15","15-2","16","16-2","18","18-2","15-3"]
for i in captures:
    data = np.load(f'/mnt/grafos_npz2/graph_201108{i}.npz', allow_pickle=True)
    output.append(Graph(x=data['x'], a=data['a'][()], y=data['y']))


In [68]:
output

[Graph(n_nodes=605195, n_node_features=4, n_edge_features=None, n_labels=605195),
 Graph(n_nodes=440574, n_node_features=4, n_edge_features=None, n_labels=440574),
 Graph(n_nodes=430265, n_node_features=4, n_edge_features=None, n_labels=430265),
 Graph(n_nodes=184901, n_node_features=4, n_edge_features=None, n_labels=184901),
 Graph(n_nodes=41399, n_node_features=4, n_edge_features=None, n_labels=41399),
 Graph(n_nodes=106580, n_node_features=4, n_edge_features=None, n_labels=106580),
 Graph(n_nodes=37943, n_node_features=4, n_edge_features=None, n_labels=37943),
 Graph(n_nodes=196686, n_node_features=4, n_edge_features=None, n_labels=196686),
 Graph(n_nodes=41712, n_node_features=4, n_edge_features=None, n_labels=41712),
 Graph(n_nodes=313678, n_node_features=4, n_edge_features=None, n_labels=313678)]

In [54]:
data = np.load('/mnt/grafos_npz2/graph_20110810.npz', allow_pickle=True)


In [66]:
Graph(x=data['x'], a=data['a'][()], y=data['y']) # a=data['a'].item()

Graph(n_nodes=605195, n_node_features=4, n_edge_features=None, n_labels=605195)

In [43]:
Graph(x=data['x'], a=data['a'], y=data['y'])

ValueError: a must have shape (n_nodes, n_nodes), got rank 0

In [5]:

data = MyDataset(path="/mnt/")
#da = data.create_npz()

# Train/valid/test split
#idxs = np.random.permutation(len(data))
#split_va, split_te = int(0.8 * len(data)), int(0.9 * len(data))
#idx_tr, idx_va, idx_te = np.split(idxs, [split_va, split_te])
#data_tr = data[idx_tr]
#data_va = data[idx_va]
#data_te = data[idx_te]

# Data loaders
loader_tr = DisjointLoader(data, node_level=True, batch_size=8, epochs=2)
#loader_va = DisjointLoader(data_va, batch_size=batch_size)
#loader_te = DisjointLoader(data_te, batch_size=batch_size)


FileNotFoundError: [Errno 2] No such file or directory: '/grafos_npz/graph_20110810.npz'

In [29]:
data = MyDataset()

ValueError: a must have shape (n_nodes, n_nodes), got rank 0

In [22]:
os.listdir("/mnt/grafos_npz2")

['graph_20110810.npz',
 'graph_20110812.npz',
 'graph_20110811.npz',
 'graph_20110816-2.npz',
 'graph_20110817.npz',
 'graph_20110815.npz',
 'graph_20110816.npz',
 'graph_20110816-3.npz',
 'graph_20110815-3.npz',
 'graph_20110815-2.npz',
 'graph_20110818-2.npz',
 'graph_20110818.npz',
 'graph_20110819.npz']

In [71]:
# Training
training_grafos = pd.read_csv(
    "/mnt/primeras-pruebas/edges_int/edges_INT_capture20110810.ncol",
    sep=" ",  
    header=None,  # no heading row
    names=["source", "target", "weight"],  # set our own names for the columns
)

training_features = pd.read_csv(
    "/mnt/primeras-pruebas/features_int/features_INT_capture20110810.csv",
    sep=",",  
    header=0
)

# Create an edges array (adjacency matrix) of shape [2, num_edges]
training_edges = training_grafos[["source", "target"]].to_numpy().T
#validation_edges = validation_grafos[["source", "target"]].to_numpy().T

# Create an edge weights array.
training_edge_weights = training_grafos[["weight"]].to_numpy().T 
training_edge_weights = training_edge_weights.reshape((training_edges.shape[-1],))
training_edge_weights = tf.convert_to_tensor(training_edge_weights)

# Create a node features array of shape [num_nodes, num_features].
feature_names = set(training_features.columns) - {"node", "label"}
training_node_features = tf.cast(
    training_features.sort_values("node")[feature_names].to_numpy(), dtype=tf.dtypes.float32
)

# Create graph info tuple with node_features, edges, and edge_weights.
graph_info = (training_node_features, training_edges, training_edge_weights)

print("Edges shape:", training_edges.shape)
print("Nodes shape:", training_node_features.shape)


Edges shape: (2, 1234211)
Nodes shape: (605195, 4)


<ipython-input-71-7419e230b593>:27: FutureWarning: Passing a set as an indexer is deprecated and will raise in a future version. Use a list instead.
  training_features.sort_values("node")[feature_names].to_numpy(), dtype=tf.dtypes.float32


In [ ]:
feature_names = set(training_features.columns) - {"node", "label"}
training_node_features = tf.cast(
    training_features.sort_values("node")[feature_names].to_numpy(), dtype=tf.dtypes.float32
)


In [97]:
training_node_features2 = tf.cast(
    training_features.sort_values("node")[training_features.columns.difference(["node","label"], sort=False)].to_numpy(), dtype=tf.dtypes.float32
)
#, sort=False

In [99]:
#training_features.loc[:,training_features.columns!={"node","label"}]
training_node_features == training_node_features2

<tf.Tensor: shape=(605195, 4), dtype=bool, numpy=
array([[False,  True,  True, False],
       [False, False,  True,  True],
       [False, False,  True,  True],
       ...,
       [False,  True,  True, False],
       [False,  True,  True, False],
       [False, False,  True, False]])>

In [100]:
training_node_features2

<tf.Tensor: shape=(605195, 4), dtype=float32, numpy=
array([[0.000e+00, 0.000e+00, 1.000e+00, 2.239e+03],
       [1.000e+00, 6.000e+00, 1.000e+00, 6.000e+00],
       [1.000e+00, 2.000e+00, 1.000e+00, 2.000e+00],
       ...,
       [0.000e+00, 0.000e+00, 1.000e+00, 6.000e+00],
       [0.000e+00, 0.000e+00, 1.000e+00, 1.200e+01],
       [3.000e+00, 2.400e+01, 0.000e+00, 0.000e+00]], dtype=float32)>

In [101]:
training_node_features

<tf.Tensor: shape=(605195, 4), dtype=float32, numpy=
array([[2.239e+03, 0.000e+00, 1.000e+00, 0.000e+00],
       [6.000e+00, 1.000e+00, 1.000e+00, 6.000e+00],
       [2.000e+00, 1.000e+00, 1.000e+00, 2.000e+00],
       ...,
       [6.000e+00, 0.000e+00, 1.000e+00, 0.000e+00],
       [1.200e+01, 0.000e+00, 1.000e+00, 0.000e+00],
       [0.000e+00, 3.000e+00, 0.000e+00, 2.400e+01]], dtype=float32)>

In [94]:
training_features.sort_values("node")[training_features.columns.difference(["node","label"], sort=False)]

,ID,OD,IDW,ODW
0,0,1,0,2239
2,1,1,6,6
4,1,1,2,2
5,1,1,1,1
6,1,1,1,1
...,...,...,...,...
557752,1,1,8,8
605191,0,1,0,6
605193,0,1,0,6
605194,0,1,0,12


In [96]:
training_features.sort_values("node")[feature_names]

<ipython-input-96-285c912f1276>:1: FutureWarning: Passing a set as an indexer is deprecated and will raise in a future version. Use a list instead.
  training_features.sort_values("node")[feature_names]


,ODW,ID,OD,IDW
0,2239,0,1,0
2,6,1,1,6
4,2,1,1,2
5,1,1,1,1
6,1,1,1,1
...,...,...,...,...
557752,8,1,1,8
605191,6,0,1,0
605193,6,0,1,0
605194,12,0,1,0


In [95]:
feature_names

{'ID', 'IDW', 'OD', 'ODW'}

In [58]:
b = np.maximum(a, a.T).astype(int)
b

array([[0, 1, 0],
       [1, 0, 0],
       [0, 0, 1]])

In [55]:
import scipy.sparse as sp
sp.csr_matrix(a)

<3x3 sparse matrix of type '<class 'numpy.int64'>'
	with 2 stored elements in Compressed Sparse Row format>

In [59]:
np.maximum(a, a.T)

array([[False,  True, False],
       [ True, False, False],
       [False, False,  True]])

In [64]:
c=np.random.randint(0,2,(3,3,1))
print(c.shape)
c

(3, 3, 1)


array([[[0],
        [1],
        [0]],

       [[1],
        [1],
        [0]],

       [[1],
        [1],
        [0]]])

In [207]:
nodos=[]
for i in range(1113):
    nodos.append(dataset[i].n_nodes)
    
dataset[np.argmin(nodos)].y #a.todense()

array([1., 0.])

In [69]:
row = np.array([0, 3, 1, 0])

col = np.array([0, 3, 1, 2])

data = np.array([4, 5, 7, 9])

mtx = sp.coo_matrix((data, (row, col)), shape=(4, 4))
mtx.todense()

matrix([[4, 0, 9, 0],
        [0, 7, 0, 0],
        [0, 0, 0, 0],
        [0, 0, 0, 5]])

In [141]:
row = np.array([0, 3, 1, 0, 2,2])

col = np.array([0, 3, 1, 2, 0,2])

data = np.array([4, 5, 7, 9, 1,99])

mtx = sp.coo_matrix((data, (row, col)), shape=(4, 4))
mtx.todense()

matrix([[ 4,  0,  9,  0],
        [ 0,  7,  0,  0],
        [ 1,  0, 99,  0],
        [ 0,  0,  0,  5]])

In [87]:
os.path.join(path,"grafos_npz")

NameError: name 'path' is not defined

In [112]:
os.listdir("/mnt/edges_int")

['edges_INT_capture20110818-2.ncol',
 'edges_INT_capture20110816-2.ncol',
 'edges_INT_capture20110812.ncol',
 'edges_INT_capture20110810.ncol',
 'edges_INT_capture20110815-2.ncol',
 'edges_INT_capture20110818.ncol',
 'edges_INT_capture20110811.ncol',
 'edges_INT_capture20110816.ncol',
 'edges_INT_capture20110815.ncol',
 'edges_INT_capture20110815-3.ncol']

In [145]:
np.reshape(training_grafos[["target"]].to_numpy().T,1234211).shape

(1234211,)

In [204]:
a_tmp = pd.read_csv('/mnt/edges_int/edges_INT_capture20110811.ncol', sep=" ", header=None, names=["source", "target", "weight"])
a_source_tmp = a_tmp[["source"]].to_numpy().T
a_source = np.reshape(a_source_tmp, a_source_tmp.shape[-1])
a_target_tmp = a_tmp[["target"]].to_numpy().T
a_target = np.reshape(a_target_tmp, a_target_tmp.shape[-1])
a_weight_tmp = a_tmp[["weight"]].to_numpy().T
a_weight = np.reshape(a_weight_tmp, a_weight_tmp.shape[-1])
a = sparse.coo_matrix((a_weight, (a_source, a_target)), shape=(440574,440574))


ValueError: row index exceeds matrix dimensions

In [203]:
len(pd.concat([a_tmp["source"],a_tmp["target"]]).unique())

440574

In [194]:
len(a_source.unique())

AttributeError: 'numpy.ndarray' object has no attribute 'unique'

In [184]:
training_int = pd.read_csv(
    "/mnt/training_GRAFOS_INT.pkts.ncol",
    sep=" ",  
    header=None,  # no heading row
    names=["source", "target", "weight"],  # set our own names for the columns
)


In [69]:
training_int[605194:]

NameError: name 'training_int' is not defined

In [28]:
from spektral.datasets import TUDataset
dataset = TUDataset("PROTEINS")

/usr/local/lib/python3.8/dist-packages/sklearn/preprocessing/_encoders.py:868: FutureWarning: `sparse` was renamed to `sparse_output` in version 1.2 and will be removed in 1.4. `sparse_output` is ignored unless you leave `sparse` to its default value.
  warnings.warn(


Successfully loaded PROTEINS.


/usr/local/lib/python3.8/dist-packages/sklearn/preprocessing/_encoders.py:868: FutureWarning: `sparse` was renamed to `sparse_output` in version 1.2 and will be removed in 1.4. `sparse_output` is ignored unless you leave `sparse` to its default value.
  warnings.warn(


In [34]:
from spektral.data.utils import prepend_none
prepend_none(dataset.signature["y"]["shape"])

(None, 2)

In [35]:
loader_train = DisjointLoader(dataset, batch_size=2, epochs=200, shuffle=False)

In [36]:
dataset.signature

{'x': {'spec': tensorflow.python.framework.tensor_spec.TensorSpec,
  'shape': (None, 4),
  'dtype': tf.float64},
 'a': {'spec': tensorflow.python.framework.sparse_tensor.SparseTensorSpec,
  'shape': (None, None),
  'dtype': tf.float64},
 'y': {'spec': tensorflow.python.framework.tensor_spec.TensorSpec,
  'shape': (2,),
  'dtype': tf.float64}}

In [15]:
import numpy as np
from scipy import sparse
row  = np.array([0, 0, 1, 3, 1, 0, 0])

col  = np.array([0, 2, 1, 3, 1, 0, 0])

data = np.array([1, 1, 1, 1, 1, 1, 1])

#coo = sparse.coo_matrix((data, (row, col)), shape=(4, 4))
coo = sparse.csr_matrix((data, (row, col)), shape=(4, 4))
coo

<4x4 sparse matrix of type '<class 'numpy.int64'>'
	with 4 stored elements in Compressed Sparse Row format>

In [26]:
for batch in loader_train:
    print(batch)

((array([[ 8.,  1.,  0.,  0.],
       [ 8.,  1.,  0.,  0.],
       [17.,  1.,  0.,  0.],
       [13.,  1.,  0.,  0.],
       [10.,  1.,  0.,  0.],
       [ 9.,  1.,  0.,  0.],
       [16.,  1.,  0.,  0.],
       [ 3.,  1.,  0.,  0.],
       [16.,  1.,  0.,  0.],
       [ 3.,  0.,  1.,  0.],
       [11.,  1.,  0.,  0.],
       [ 7.,  1.,  0.,  0.],
       [ 3.,  1.,  0.,  0.],
       [ 4.,  1.,  0.,  0.],
       [11.,  1.,  0.,  0.],
       [ 7.,  1.,  0.,  0.],
       [ 3.,  1.,  0.,  0.],
       [ 4.,  1.,  0.,  0.],
       [11.,  1.,  0.,  0.],
       [ 7.,  1.,  0.,  0.],
       [ 3.,  1.,  0.,  0.],
       [ 4.,  1.,  0.,  0.],
       [ 4.,  0.,  1.,  0.],
       [ 5.,  0.,  1.,  0.],
       [ 5.,  0.,  1.,  0.],
       [ 4.,  0.,  1.,  0.],
       [ 4.,  0.,  1.,  0.],
       [ 5.,  0.,  1.,  0.],
       [ 5.,  0.,  1.,  0.],
       [ 5.,  0.,  1.,  0.],
       [ 4.,  0.,  1.,  0.],
       [ 5.,  0.,  1.,  0.],
       [ 5.,  0.,  1.,  0.],
       [ 4.,  0.,  1.,  0.]]), <tensorflo

((array([[15.,  1.,  0.,  0.],
       [18.,  1.,  0.,  0.],
       [10.,  1.,  0.,  0.],
       [10.,  1.,  0.,  0.],
       [13.,  1.,  0.,  0.],
       [14.,  1.,  0.,  0.],
       [ 9.,  1.,  0.,  0.],
       [ 3.,  1.,  0.,  0.],
       [ 7.,  1.,  0.,  0.],
       [ 3.,  1.,  0.,  0.],
       [ 6.,  1.,  0.,  0.],
       [ 3.,  1.,  0.,  0.],
       [14.,  1.,  0.,  0.],
       [ 3.,  0.,  1.,  0.],
       [ 4.,  0.,  1.,  0.],
       [ 4.,  0.,  1.,  0.],
       [ 5.,  0.,  1.,  0.],
       [ 7.,  0.,  1.,  0.],
       [ 5.,  0.,  1.,  0.],
       [ 3.,  0.,  1.,  0.],
       [ 4.,  0.,  1.,  0.],
       [ 3.,  0.,  1.,  0.],
       [14.,  1.,  0.,  0.],
       [ 6.,  1.,  0.,  0.],
       [11.,  1.,  0.,  0.],
       [ 9.,  1.,  0.,  0.],
       [ 9.,  1.,  0.,  0.],
       [ 8.,  1.,  0.,  0.],
       [13.,  1.,  0.,  0.],
       [16.,  1.,  0.,  0.],
       [ 6.,  1.,  0.,  0.],
       [ 4.,  1.,  0.,  0.],
       [ 3.,  1.,  0.,  0.],
       [ 3.,  1.,  0.,  0.],
       [ 3.,

((array([[ 3.,  1.,  0.,  0.],
       [ 6.,  1.,  0.,  0.],
       [ 8.,  1.,  0.,  0.],
       [16.,  1.,  0.,  0.],
       [15.,  1.,  0.,  0.],
       [ 8.,  1.,  0.,  0.],
       [ 6.,  1.,  0.,  0.],
       [17.,  1.,  0.,  0.],
       [15.,  1.,  0.,  0.],
       [ 5.,  1.,  0.,  0.],
       [ 9.,  1.,  0.,  0.],
       [ 3.,  0.,  1.,  0.],
       [ 3.,  0.,  1.,  0.],
       [ 5.,  0.,  1.,  0.],
       [10.,  0.,  1.,  0.],
       [11.,  0.,  1.,  0.],
       [ 7.,  0.,  1.,  0.],
       [ 7.,  0.,  1.,  0.],
       [ 3.,  0.,  1.,  0.],
       [ 9.,  0.,  1.,  0.],
       [ 8.,  0.,  1.,  0.],
       [ 3.,  0.,  1.,  0.],
       [ 3.,  0.,  1.,  0.],
       [ 2.,  0.,  1.,  0.],
       [ 6.,  0.,  1.,  0.],
       [ 8.,  0.,  1.,  0.],
       [ 8.,  0.,  1.,  0.],
       [ 3.,  0.,  1.,  0.],
       [ 3.,  0.,  1.,  0.],
       [14.,  1.,  0.,  0.],
       [ 3.,  1.,  0.,  0.],
       [15.,  1.,  0.,  0.],
       [11.,  1.,  0.,  0.],
       [14.,  1.,  0.,  0.],
       [ 4.,

((array([[ 5.,  1.,  0.,  0.],
       [ 6.,  1.,  0.,  0.],
       [16.,  1.,  0.,  0.],
       [16.,  1.,  0.,  0.],
       [ 4.,  1.,  0.,  0.],
       [ 5.,  1.,  0.,  0.],
       [11.,  1.,  0.,  0.],
       [15.,  1.,  0.,  0.],
       [11.,  1.,  0.,  0.],
       [16.,  1.,  0.,  0.],
       [ 5.,  1.,  0.,  0.],
       [ 6.,  1.,  0.,  0.],
       [16.,  1.,  0.,  0.],
       [16.,  1.,  0.,  0.],
       [ 4.,  1.,  0.,  0.],
       [ 5.,  1.,  0.,  0.],
       [12.,  1.,  0.,  0.],
       [15.,  1.,  0.,  0.],
       [11.,  1.,  0.,  0.],
       [16.,  1.,  0.,  0.],
       [ 2.,  0.,  1.,  0.],
       [ 4.,  0.,  1.,  0.],
       [ 7.,  0.,  1.,  0.],
       [10.,  0.,  1.,  0.],
       [10.,  0.,  1.,  0.],
       [ 3.,  0.,  1.,  0.],
       [10.,  0.,  1.,  0.],
       [ 5.,  0.,  1.,  0.],
       [ 4.,  0.,  1.,  0.],
       [ 3.,  0.,  1.,  0.],
       [ 3.,  0.,  1.,  0.],
       [ 3.,  0.,  1.,  0.],
       [ 4.,  0.,  1.,  0.],
       [ 5.,  0.,  1.,  0.],
       [ 8.,

((array([[19.,  1.,  0.,  0.],
       [17.,  1.,  0.,  0.],
       [ 3.,  1.,  0.,  0.],
       ...,
       [ 6.,  0.,  1.,  0.],
       [ 5.,  0.,  1.,  0.],
       [ 5.,  0.,  1.,  0.]]), <tensorflow.python.framework.sparse_tensor.SparseTensor object at 0x7f052bf050d0>, array([0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
       0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
       1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
       1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
       1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
       1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
       1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
       1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
       1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
       1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 

...

((array([[ 3.,  1.,  0.,  0.],
       [ 9.,  0.,  1.,  0.],
       [ 9.,  0.,  1.,  0.],
       [ 8.,  0.,  1.,  0.],
       [ 3.,  0.,  1.,  0.],
       [ 6.,  0.,  1.,  0.],
       [ 8.,  0.,  1.,  0.],
       [ 8.,  0.,  1.,  0.],
       [19.,  1.,  0.,  0.],
       [ 8.,  1.,  0.,  0.],
       [ 9.,  1.,  0.,  0.],
       [ 4.,  1.,  0.,  0.],
       [ 3.,  1.,  0.,  0.],
       [11.,  1.,  0.,  0.],
       [ 3.,  1.,  0.,  0.],
       [ 4.,  1.,  0.,  0.],
       [ 3.,  1.,  0.,  0.],
       [17.,  1.,  0.,  0.],
       [16.,  1.,  0.,  0.],
       [10.,  1.,  0.,  0.],
       [ 3.,  1.,  0.,  0.],
       [13.,  1.,  0.,  0.],
       [ 3.,  1.,  0.,  0.],
       [ 3.,  1.,  0.,  0.],
       [13.,  1.,  0.,  0.],
       [17.,  1.,  0.,  0.],
       [ 3.,  1.,  0.,  0.],
       [11.,  1.,  0.,  0.],
       [ 4.,  0.,  1.,  0.],
       [ 5.,  0.,  1.,  0.],
       [ 4.,  0.,  1.,  0.],
       [ 5.,  0.,  1.,  0.],
       [ 6.,  0.,  1.,  0.],
       [ 9.,  0.,  1.,  0.],
       [ 7.,

((array([[16.,  1.,  0.,  0.],
       [12.,  1.,  0.,  0.],
       [14.,  1.,  0.,  0.],
       [14.,  1.,  0.,  0.],
       [13.,  1.,  0.,  0.],
       [ 3.,  1.,  0.,  0.],
       [10.,  1.,  0.,  0.],
       [ 3.,  1.,  0.,  0.],
       [ 5.,  1.,  0.,  0.],
       [ 3.,  1.,  0.,  0.],
       [16.,  1.,  0.,  0.],
       [ 3.,  1.,  0.,  0.],
       [ 4.,  1.,  0.,  0.],
       [15.,  1.,  0.,  0.],
       [15.,  1.,  0.,  0.],
       [13.,  1.,  0.,  0.],
       [13.,  1.,  0.,  0.],
       [15.,  1.,  0.,  0.],
       [13.,  1.,  0.,  0.],
       [ 4.,  1.,  0.,  0.],
       [ 3.,  1.,  0.,  0.],
       [13.,  1.,  0.,  0.],
       [18.,  1.,  0.,  0.],
       [ 4.,  1.,  0.,  0.],
       [14.,  1.,  0.,  0.],
       [13.,  1.,  0.,  0.],
       [14.,  1.,  0.,  0.],
       [13.,  1.,  0.,  0.],
       [ 4.,  1.,  0.,  0.],
       [ 3.,  1.,  0.,  0.],
       [13.,  1.,  0.,  0.],
       [13.,  1.,  0.,  0.],
       [17.,  1.,  0.,  0.],
       [16.,  1.,  0.,  0.],
       [12.,

((array([[ 8.,  1.,  0.,  0.],
       [16.,  1.,  0.,  0.],
       [14.,  1.,  0.,  0.],
       [11.,  1.,  0.,  0.],
       [ 9.,  1.,  0.,  0.],
       [12.,  1.,  0.,  0.],
       [ 7.,  1.,  0.,  0.],
       [12.,  1.,  0.,  0.],
       [21.,  1.,  0.,  0.],
       [ 6.,  0.,  1.,  0.],
       [ 5.,  0.,  1.,  0.],
       [16.,  0.,  1.,  0.],
       [ 9.,  0.,  1.,  0.],
       [ 6.,  0.,  1.,  0.],
       [ 5.,  0.,  1.,  0.],
       [ 7.,  0.,  1.,  0.],
       [ 7.,  0.,  1.,  0.],
       [ 3.,  0.,  1.,  0.],
       [18.,  1.,  0.,  0.],
       [12.,  1.,  0.,  0.],
       [18.,  1.,  0.,  0.],
       [ 8.,  1.,  0.,  0.],
       [22.,  1.,  0.,  0.],
       [ 4.,  1.,  0.,  0.],
       [14.,  1.,  0.,  0.],
       [ 3.,  1.,  0.,  0.],
       [14.,  1.,  0.,  0.],
       [ 9.,  1.,  0.,  0.],
       [ 4.,  1.,  0.,  0.],
       [ 9.,  1.,  0.,  0.],
       [ 4.,  1.,  0.,  0.],
       [13.,  1.,  0.,  0.],
       [ 9.,  1.,  0.,  0.],
       [17.,  1.,  0.,  0.],
       [ 9.,

((array([[13.,  1.,  0.,  0.],
       [13.,  1.,  0.,  0.],
       [13.,  1.,  0.,  0.],
       [13.,  1.,  0.,  0.],
       [13.,  1.,  0.,  0.],
       [ 5.,  0.,  1.,  0.],
       [ 5.,  0.,  1.,  0.],
       [ 6.,  0.,  1.,  0.],
       [ 5.,  0.,  1.,  0.],
       [ 5.,  0.,  1.,  0.],
       [ 6.,  0.,  1.,  0.],
       [ 5.,  0.,  1.,  0.],
       [ 5.,  0.,  1.,  0.],
       [ 6.,  0.,  1.,  0.],
       [ 5.,  0.,  1.,  0.],
       [ 5.,  0.,  1.,  0.],
       [ 6.,  0.,  1.,  0.],
       [ 4.,  0.,  1.,  0.],
       [ 5.,  0.,  1.,  0.],
       [ 5.,  0.,  1.,  0.],
       [ 5.,  0.,  1.,  0.],
       [ 5.,  0.,  1.,  0.],
       [ 6.,  0.,  1.,  0.],
       [ 5.,  0.,  1.,  0.],
       [ 5.,  0.,  1.,  0.],
       [ 6.,  0.,  1.,  0.],
       [ 5.,  0.,  1.,  0.],
       [ 5.,  0.,  1.,  0.],
       [ 6.,  0.,  1.,  0.],
       [ 5.,  0.,  1.,  0.],
       [ 5.,  0.,  1.,  0.],
       [ 6.,  0.,  1.,  0.],
       [ 5.,  0.,  1.,  0.],
       [ 5.,  0.,  1.,  0.],
       [ 6.,

KeyboardInterrupt: 

In [16]:
coo.todense()

matrix([[3, 0, 1, 0],
        [0, 2, 0, 0],
        [0, 0, 0, 0],
        [0, 0, 0, 1]])

In [17]:
coo2 = sparse.coo_matrix((data, (row, col)), shape=(4, 4))
coo2.todense()

matrix([[3, 0, 1, 0],
        [0, 2, 0, 0],
        [0, 0, 0, 0],
        [0, 0, 0, 1]])

In [20]:
import tensorflow as tf
tf.convert_to_tensor(coo)

ValueError: TypeError: sparse matrix length is ambiguous; use getnnz() or shape[0]
Traceback (most recent call last):

  File "/usr/local/lib/python3.8/dist-packages/scipy/sparse/_base.py", line 345, in __len__
    raise TypeError("sparse matrix length is ambiguous; use getnnz()"

TypeError: sparse matrix length is ambiguous; use getnnz() or shape[0]



In [36]:
import shutil 

shutil.rmtree("/root/spektral/datasets/CTU13balancedODWbytes") 

In [31]:
import os
os.listdir("/root/spektral/datasets/")

['CTU13balancedODW',
 'CTU13random',
 'CTU13balancedIDOD',
 'MisProteinas',
 'TUDataset',
 'CTU13',
 'CTU13grafos']

In [3]:
import os
os.listdir("/root/spektral/datasets/")

['CTU131', 'CTU13random', 'MisProteinas', 'TUDataset', 'CTU13', 'CTU13grafos']

In [2]:
os.rename("/root/spektral/datasets/CTU13","/root/spektral/datasets/CTU131/")

In [6]:
import pandas as pd
df=pd.read_csv("https://nube.ingenieria.uncuyo.edu.ar/s/ktAgJS22kBXcMsT/download",header=0)

In [11]:
import filecmp
result = filecmp.dircmp("/root/spektral/datasets/CTU13/","/root/spektral/datasets/CTU131/")
result.report()

diff /root/spektral/datasets/CTU13/ /root/spektral/datasets/CTU131/
Identical files : ['graph_20110810.npz', 'graph_20110811.npz', 'graph_20110812.npz', 'graph_20110815-2.npz', 'graph_20110815-3.npz', 'graph_20110815.npz', 'graph_20110816-2.npz', 'graph_20110816-3.npz', 'graph_20110816.npz', 'graph_20110817.npz', 'graph_20110818-2.npz', 'graph_20110818.npz', 'graph_20110819.npz']


####################

In [1]:
import numpy as np
import tensorflow as tf
from tensorflow.keras.layers import Dense, Dropout,Input
from tensorflow.keras.losses import CategoricalCrossentropy
from tensorflow.keras.metrics import categorical_accuracy
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import Adam
from spektral.layers import GINConv,GCNConv
from spektral.utils.sparse import sp_matrix_to_sp_tensor
from spektral.data import DisjointLoader, BatchLoader
from spektral.datasets import TUDataset

In [2]:
dataset = TUDataset("PROTEINS")

/usr/local/lib/python3.8/dist-packages/sklearn/preprocessing/_encoders.py:868: FutureWarning: `sparse` was renamed to `sparse_output` in version 1.2 and will be removed in 1.4. `sparse_output` is ignored unless you leave `sparse` to its default value.
  warnings.warn(


Successfully loaded PROTEINS.


/usr/local/lib/python3.8/dist-packages/sklearn/preprocessing/_encoders.py:868: FutureWarning: `sparse` was renamed to `sparse_output` in version 1.2 and will be removed in 1.4. `sparse_output` is ignored unless you leave `sparse` to its default value.
  warnings.warn(


In [4]:
dataset[0]

Graph(n_nodes=42, n_node_features=4, n_edge_features=None, n_labels=2)

In [5]:
split = int(0.8 * len(dataset))
dataset_train, dataset_test = dataset[:split], dataset[split:]


In [6]:
batch_size = 32
loader_train = DisjointLoader(dataset_train, node_level=True, batch_size=batch_size, epochs=200, shuffle=False)
loader_test = DisjointLoader(dataset_test, node_level=True, batch_size=batch_size)


In [25]:
algo=loader_train.__next__()

In [26]:
algo[0][0].shape, algo[0][1].shape, algo[0][2].shape, algo[1].shape

((1378, 4), TensorShape([1378, 1378]), (1378,), (64, 1))

In [73]:
algo[1]

array([[1.],
       [0.],
       [1.],
       [0.],
       [1.],
       [0.],
       [1.],
       [0.],
       [1.],
       [0.],
       [1.],
       [0.],
       [1.],
       [0.],
       [1.],
       [0.],
       [1.],
       [0.],
       [1.],
       [0.],
       [1.],
       [0.],
       [1.],
       [0.],
       [1.],
       [0.],
       [1.],
       [0.],
       [1.],
       [0.],
       [1.],
       [0.],
       [1.],
       [0.],
       [1.],
       [0.],
       [1.],
       [0.],
       [1.],
       [0.],
       [1.],
       [0.],
       [1.],
       [0.],
       [1.],
       [0.],
       [1.],
       [0.],
       [1.],
       [0.],
       [1.],
       [0.],
       [1.],
       [0.],
       [1.],
       [0.],
       [1.],
       [0.],
       [1.],
       [0.],
       [1.],
       [0.],
       [1.],
       [0.]])

In [2]:
import os
os.listdir("/root/spektral/datasets/") # README.md`

['CTU131', 'CTU13random', 'MisProteinas', 'TUDataset', 'CTU13', 'CTU13grafos']

In [1]:
os.listdir("/root/spektral/datasets/")

NameError: name 'os' is not defined

In [ ]:
import glob
import os
import shutil
from os import path as osp
from urllib.error import URLError

import numpy as np
import pandas as pd
from sklearn.preprocessing import OneHotEncoder, StandardScaler

from spektral.data import Dataset, Graph
from spektral.datasets.utils import download_file
from spektral.utils import io, sparse


class MisProteinas(Dataset):
    """
    The Benchmark Data Sets for Graph Kernels from TU Dortmund
    ([link](https://chrsmrrs.github.io/datasets/docs/datasets/)).
    Node features are computed by concatenating the following features for
    each node:
    - node attributes, if available;
    - node labels, if available, one-hot encoded.
    Some datasets might not have node features at all. In this case, attempting
    to use the dataset with a Loader will result in a crash. You can create
    node features using some of the transforms available in `spektral.transforms`
    or you can define your own features by accessing the individual samples in
    the `graph` attribute of the dataset (which is a list of `Graph` objects).
    Edge features are computed by concatenating the following features for
    each node:
    - edge attributes, if available;
    - edge labels, if available, one-hot encoded.
    Graph labels are provided for each dataset.
    Specific details about each individual dataset can be found in
    `~/spektral/datasets/TUDataset/<dataset name>/README.md`, after the dataset
    has been downloaded locally (datasets are downloaded automatically upon
    calling `TUDataset('<dataset name>')` the first time).
    **Arguments**
    - `name`: str, name of the dataset to load (see `TUD.available_datasets`).
    - `clean`: if `True`, rload a version of the dataset with no isomorphic
               graphs.
    """

    url = "https://www.chrsmrrs.com/graphkerneldatasets"
    url_clean = (
        "https://raw.githubusercontent.com/nd7141/graph_datasets/master/datasets"
    )

    def __init__(self, name, clean=False, **kwargs):
        if name not in self.available_datasets():
            raise ValueError(
                "Unknown dataset {}. See {}.available_datasets() for a complete list of"
                "available datasets.".format(name, self.__class__.__name__)
            )
        self.name = name
        self.clean = clean
        super().__init__(**kwargs)

    @property
    def path(self):
        return osp.join(super().path, self.name + ("_clean" if self.clean else ""))

    def download(self):
        print(
            "Downloading {} dataset{}.".format(
                self.name, " (clean)" if self.clean else ""
            )
        )
        url = "{}/{}.zip".format(self.url_clean if self.clean else self.url, self.name)
        download_file(url, self.path, self.name + ".zip")

        # Datasets are zipped in a folder: unpack them
        parent = self.path
        subfolder = osp.join(self.path, self.name)
        for filename in os.listdir(subfolder):
            shutil.move(osp.join(subfolder, filename), osp.join(parent, filename))
        os.rmdir(subfolder)

    def read(self):
        fname_template = osp.join(self.path, "{}_{{}}.txt".format(self.name))
        available = [
            f.split(os.sep)[-1][len(self.name) + 1 : -4]  # Remove leading name
            for f in glob.glob(fname_template.format("*"))
        ]

        # Batch index
        node_batch_index = (
            io.load_txt(fname_template.format("graph_indicator")).astype(int) - 1
        )
        n_nodes = np.bincount(node_batch_index)
        n_nodes_cum = np.concatenate(([0], np.cumsum(n_nodes)[:-1]))

        # Read edge lists
        edges = io.load_txt(fname_template.format("A"), delimiter=",").astype(int) - 1
        # Remove duplicates and self-loops from edges
        _, mask = np.unique(edges, axis=0, return_index=True)
        mask = mask[edges[mask, 0] != edges[mask, 1]]
        edges = edges[mask]
        # Split edges into separate edge lists
        edge_batch_idx = node_batch_index[edges[:, 0]]
        n_edges = np.bincount(edge_batch_idx)
        n_edges_cum = np.cumsum(n_edges[:-1])
        el_list = np.split(edges - n_nodes_cum[edge_batch_idx, None], n_edges_cum)

        # Node features
        x_list = []
        if "node_attributes" in available:
            x_attr = io.load_txt(
                fname_template.format("node_attributes"), delimiter=","
            )
            if x_attr.ndim == 1:
                x_attr = x_attr[:, None]
            x_list.append(x_attr)
        #if "node_labels" in available:
        #    x_labs = io.load_txt(fname_template.format("node_labels"))
        #    if x_labs.ndim == 1:
        #        x_labs = x_labs[:, None]
        #    x_labs = np.concatenate(
        #        [_normalize(xl_[:, None], "ohe") for xl_ in x_labs.T], -1
        #    )
        #    x_list.append(x_labs)
        if len(x_list) > 0:
            x_list = np.concatenate(x_list, -1)
            x_list = np.split(x_list, n_nodes_cum[1:])
        else:
            print(
                "WARNING: this dataset doesn't have node attributes."
                "Consider creating manual features before using it with a "
                "Loader."
            )
            x_list = [None] * len(n_nodes)

        # Edge features
        e_list = []
        if "edge_attributes" in available:
            e_attr = io.load_txt(fname_template.format("edge_attributes"))
            if e_attr.ndim == 1:
                e_attr = e_attr[:, None]
            e_attr = e_attr[mask]
            e_list.append(e_attr)
        if "edge_labels" in available:
            e_labs = io.load_txt(fname_template.format("edge_labels"))
            if e_labs.ndim == 1:
                e_labs = e_labs[:, None]
            e_labs = e_labs[mask]
            e_labs = np.concatenate(
                [_normalize(el_[:, None], "ohe") for el_ in e_labs.T], -1
            )
            e_list.append(e_labs)
        if len(e_list) > 0:
            e_available = True
            e_list = np.concatenate(e_list, -1)
            e_list = np.split(e_list, n_edges_cum)
        else:
            e_available = False
            e_list = [None] * len(n_nodes)

        # Create sparse adjacency matrices and re-sort edge attributes in lexicographic
        # order
        a_e_list = [
            sparse.edge_index_to_matrix(
                edge_index=el,
                edge_weight=np.ones(el.shape[0]),
                edge_features=e,
                shape=(n, n),
            )
            for el, e, n in zip(el_list, e_list, n_nodes)
        ]
        if e_available:
            a_list, e_list = list(zip(*a_e_list))
        else:
            a_list = a_e_list

        # Labels
        if "node_labels" in available:
            labels = io.load_txt(fname_template.format("node_labels"))
            labels = _normalize(labels[:, None], "ohe")
            #if x_labs.ndim == 1:
            #    x_labs = x_labs[:, None]
            #x_labs = np.concatenate(
            #    [_normalize(xl_[:, None], "ohe") for xl_ in x_labs.T], -1
            #)
        #if "graph_attributes" in available:
        #    labels = io.load_txt(fname_template.format("graph_attributes"))
        #elif "graph_labels" in available:
        #    labels = io.load_txt(fname_template.format("graph_labels"))
        #    labels = _normalize(labels[:, None], "ohe")
        else:
            raise ValueError("No labels available for dataset {}".format(self.name))

        # Convert to Graph
        print("Successfully loaded {}.".format(self.name))
        return [
            Graph(x=x, a=a, e=e, y=y)
            for x, a, e, y in zip(x_list, a_list, e_list, labels)
        ]

    @staticmethod
    def available_datasets():
        url = "https://chrsmrrs.github.io/datasets/docs/datasets/"
        try:
            tables = pd.read_html(url)
            names = []
            for table in tables:
                names.extend(table.Name[1:].values.tolist())
            return names
        except URLError:
            # No internet, don't panic
            print("Could not read URL {}".format(url))
            return []


def _normalize(x, norm=None):
    """
    Apply one-hot encoding or z-score to a list of node features
    """
    if norm == "ohe":
        fnorm = OneHotEncoder(sparse=False, categories="auto")
    elif norm == "zscore":
        fnorm = StandardScaler()
    else:
        return x
    return fnorm.fit_transform(x)


In [2]:
import glob
import os
import shutil
from os import path as osp
from urllib.error import URLError

import numpy as np
import pandas as pd
from sklearn.preprocessing import OneHotEncoder, StandardScaler

from spektral.data import Dataset, Graph
from spektral.datasets.utils import download_file
from spektral.utils import io, sparse


In [36]:
name = "PROTEINS"
path = "/root/spektral/datasets/TUDataset/PROTEINS"
fname_template = osp.join(path, "{}_{{}}.txt".format(name))
available = [
            f.split(os.sep)[-1][len(name) + 1 : -4]  # Remove leading name
            for f in glob.glob(fname_template.format("*"))
]

In [37]:
available

['graph_indicator', 'node_attributes', 'A', 'node_labels', 'graph_labels']

In [38]:
# Batch index
node_batch_index = (
            io.load_txt(fname_template.format("graph_indicator")).astype(int) - 1
)
n_nodes = np.bincount(node_batch_index)
n_nodes_cum = np.concatenate(([0], np.cumsum(n_nodes)[:-1]))

In [59]:
# Read edge lists
edges = io.load_txt(fname_template.format("A"), delimiter=",").astype(int) - 1
# Remove duplicates and self-loops from edges
_, mask = np.unique(edges, axis=0, return_index=True)
mask = mask[edges[mask, 0] != edges[mask, 1]]
edges = edges[mask]
# Split edges into separate edge lists
edge_batch_idx = node_batch_index[edges[:, 0]]
n_edges = np.bincount(edge_batch_idx)
n_edges_cum = np.cumsum(n_edges[:-1])
el_list = np.split(edges - n_nodes_cum[edge_batch_idx, None], n_edges_cum)


In [5]:
def _normalize(x, norm=None):
    """
    Apply one-hot encoding or z-score to a list of node features
    """
    if norm == "ohe":
        fnorm = OneHotEncoder(sparse=False, categories="auto")
    elif norm == "zscore":
        fnorm = StandardScaler()
    else:
        return x
    return fnorm.fit_transform(x)


In [44]:
# Node features
x_list = []
if "node_attributes" in available:
    x_attr = io.load_txt(fname_template.format("node_attributes"), delimiter=",")
    if x_attr.ndim == 1:
        x_attr = x_attr[:, None]
    x_list.append(x_attr)
if "node_labels" in available:
    x_labs = io.load_txt(fname_template.format("node_labels"))
    if x_labs.ndim == 1:
        x_labs = x_labs[:, None]
    x_labs = np.concatenate([_normalize(xl_[:, None], "ohe") for xl_ in x_labs.T], -1)
    x_list.append(x_labs)
if len(x_list) > 0:
    x_list = np.concatenate(x_list, -1)
    x_list = np.split(x_list, n_nodes_cum[1:])
else:
    print(
            "WARNING: this dataset doesn't have node attributes."
            "Consider creating manual features before using it with a "
            "Loader."
        )
    x_list = [None] * len(n_nodes)


/usr/local/lib/python3.8/dist-packages/sklearn/preprocessing/_encoders.py:868: FutureWarning: `sparse` was renamed to `sparse_output` in version 1.2 and will be removed in 1.4. `sparse_output` is ignored unless you leave `sparse` to its default value.
  warnings.warn(


In [47]:
len(x_list)

1113

In [39]:
labels = io.load_txt(fname_template.format("node_labels"))
labels = _normalize(labels[:, None], "ohe")
labels

/usr/local/lib/python3.8/dist-packages/sklearn/preprocessing/_encoders.py:868: FutureWarning: `sparse` was renamed to `sparse_output` in version 1.2 and will be removed in 1.4. `sparse_output` is ignored unless you leave `sparse` to its default value.
  warnings.warn(


array([[1., 0., 0.],
       [1., 0., 0.],
       [1., 0., 0.],
       ...,
       [0., 0., 1.],
       [0., 0., 1.],
       [0., 0., 1.]])

In [40]:
labels.shape

(43471, 3)

In [41]:
if "node_labels" in available:
    x_labs = io.load_txt(fname_template.format("node_labels"))
    if x_labs.ndim == 1:
        x_labs = x_labs[:, None]
    x_labs = np.concatenate([_normalize(xl_[:, None], "ohe") for xl_ in x_labs.T], -1)


/usr/local/lib/python3.8/dist-packages/sklearn/preprocessing/_encoders.py:868: FutureWarning: `sparse` was renamed to `sparse_output` in version 1.2 and will be removed in 1.4. `sparse_output` is ignored unless you leave `sparse` to its default value.
  warnings.warn(


In [43]:
x_labs.shape

(43471, 3)

In [ ]:
def _normalize(x, norm=None):
    """
    Apply one-hot encoding or z-score to a list of node features
    """
    if norm == "ohe":
        fnorm = OneHotEncoder(sparse=False, categories="auto")
    elif norm == "zscore":
        fnorm = StandardScaler()
    else:
        return x
    return fnorm.fit_transform(x)


In [29]:
class MisProteinas(Dataset):
    """
    The Benchmark Data Sets for Graph Kernels from TU Dortmund
    ([link](https://chrsmrrs.github.io/datasets/docs/datasets/)).
    Node features are computed by concatenating the following features for
    each node:
    - node attributes, if available;
    - node labels, if available, one-hot encoded.
    Some datasets might not have node features at all. In this case, attempting
    to use the dataset with a Loader will result in a crash. You can create
    node features using some of the transforms available in `spektral.transforms`
    or you can define your own features by accessing the individual samples in
    the `graph` attribute of the dataset (which is a list of `Graph` objects).
    Edge features are computed by concatenating the following features for
    each node:
    - edge attributes, if available;
    - edge labels, if available, one-hot encoded.
    Graph labels are provided for each dataset.
    Specific details about each individual dataset can be found in
    `~/spektral/datasets/TUDataset/<dataset name>/README.md`, after the dataset
    has been downloaded locally (datasets are downloaded automatically upon
    calling `TUDataset('<dataset name>')` the first time).
    **Arguments**
    - `name`: str, name of the dataset to load (see `TUD.available_datasets`).
    - `clean`: if `True`, rload a version of the dataset with no isomorphic
               graphs.
    """

    url = "https://www.chrsmrrs.com/graphkerneldatasets"
    url_clean = (
        "https://raw.githubusercontent.com/nd7141/graph_datasets/master/datasets"
    )

    def __init__(self, name, clean=False, **kwargs):
        if name not in self.available_datasets():
            raise ValueError(
                "Unknown dataset {}. See {}.available_datasets() for a complete list of"
                "available datasets.".format(name, self.__class__.__name__)
            )
        self.name = name
        self.clean = clean
        super().__init__(**kwargs)

    @property
    def path(self):
        return osp.join(super().path, self.name + ("_clean" if self.clean else ""))

    def download(self):
        print(
            "Downloading {} dataset{}.".format(
                self.name, " (clean)" if self.clean else ""
            )
        )
        url = "{}/{}.zip".format(self.url_clean if self.clean else self.url, self.name)
        download_file(url, self.path, self.name + ".zip")

        # Datasets are zipped in a folder: unpack them
        parent = self.path
        subfolder = osp.join(self.path, self.name)
        for filename in os.listdir(subfolder):
            shutil.move(osp.join(subfolder, filename), osp.join(parent, filename))
        os.rmdir(subfolder)

    def read(self):
        fname_template = osp.join(self.path, "{}_{{}}.txt".format(self.name))
        available = [
            f.split(os.sep)[-1][len(self.name) + 1 : -4]  # Remove leading name
            for f in glob.glob(fname_template.format("*"))
        ]

        # Batch index
        node_batch_index = (
            io.load_txt(fname_template.format("graph_indicator")).astype(int) - 1
        )
        n_nodes = np.bincount(node_batch_index)
        n_nodes_cum = np.concatenate(([0], np.cumsum(n_nodes)[:-1]))

        # Read edge lists
        edges = io.load_txt(fname_template.format("A"), delimiter=",").astype(int) - 1
        # Remove duplicates and self-loops from edges
        _, mask = np.unique(edges, axis=0, return_index=True)
        mask = mask[edges[mask, 0] != edges[mask, 1]]
        edges = edges[mask]
        # Split edges into separate edge lists
        edge_batch_idx = node_batch_index[edges[:, 0]]
        n_edges = np.bincount(edge_batch_idx)
        n_edges_cum = np.cumsum(n_edges[:-1])
        el_list = np.split(edges - n_nodes_cum[edge_batch_idx, None], n_edges_cum)

        # Node features
        x_list = []
        if "node_attributes" in available:
            x_attr = io.load_txt(
                fname_template.format("node_attributes"), delimiter=","
            )
            if x_attr.ndim == 1:
                x_attr = x_attr[:, None]
            x_list.append(x_attr)
        #if "node_labels" in available:
        #    x_labs = io.load_txt(fname_template.format("node_labels"))
        #    if x_labs.ndim == 1:
        #        x_labs = x_labs[:, None]
        #    x_labs = np.concatenate(
        #        [_normalize(xl_[:, None], "ohe") for xl_ in x_labs.T], -1
        #    )
        #    x_list.append(x_labs)
        if len(x_list) > 0:
            x_list = np.concatenate(x_list, -1)
            x_list = np.split(x_list, n_nodes_cum[1:])
        else:
            print(
                "WARNING: this dataset doesn't have node attributes."
                "Consider creating manual features before using it with a "
                "Loader."
            )
            x_list = [None] * len(n_nodes)

        # Edge features
        e_list = []
        if "edge_attributes" in available:
            e_attr = io.load_txt(fname_template.format("edge_attributes"))
            if e_attr.ndim == 1:
                e_attr = e_attr[:, None]
            e_attr = e_attr[mask]
            e_list.append(e_attr)
        if "edge_labels" in available:
            e_labs = io.load_txt(fname_template.format("edge_labels"))
            if e_labs.ndim == 1:
                e_labs = e_labs[:, None]
            e_labs = e_labs[mask]
            e_labs = np.concatenate(
                [_normalize(el_[:, None], "ohe") for el_ in e_labs.T], -1
            )
            e_list.append(e_labs)
        if len(e_list) > 0:
            e_available = True
            e_list = np.concatenate(e_list, -1)
            e_list = np.split(e_list, n_edges_cum)
        else:
            e_available = False
            e_list = [None] * len(n_nodes)

        # Create sparse adjacency matrices and re-sort edge attributes in lexicographic
        # order
        a_e_list = [
            sparse.edge_index_to_matrix(
                edge_index=el,
                edge_weight=np.ones(el.shape[0]),
                edge_features=e,
                shape=(n, n),
            )
            for el, e, n in zip(el_list, e_list, n_nodes)
        ]
        if e_available:
            a_list, e_list = list(zip(*a_e_list))
        else:
            a_list = a_e_list

        # Labels
        if "node_labels" in available:
            labels = io.load_txt(fname_template.format("node_labels"))
            if labels.ndim == 1:
                labels = labels[:, None]
            labels = np.concatenate(
                [_normalize(xl_[:, None], "ohe") for xl_ in labels.T], -1
            )
        #if "graph_attributes" in available:
        #    labels = io.load_txt(fname_template.format("graph_attributes"))
        #elif "graph_labels" in available:
        #    labels = io.load_txt(fname_template.format("graph_labels"))
        #    labels = _normalize(labels[:, None], "ohe")
        else:
            raise ValueError("No labels available for dataset {}".format(self.name))

        # Convert to Graph
        print("Successfully loaded {}.".format(self.name))
        return [
            Graph(x=x, a=a, e=e, y=y)
            for x, a, e, y in zip(x_list, a_list, e_list, labels)
        ]

    @staticmethod
    def available_datasets():
        url = "https://chrsmrrs.github.io/datasets/docs/datasets/"
        try:
            tables = pd.read_html(url)
            names = []
            for table in tables:
                names.extend(table.Name[1:].values.tolist())
            return names
        except URLError:
            # No internet, don't panic
            print("Could not read URL {}".format(url))
            return []


In [30]:
proteinas=MisProteinas("PROTEINS")

100%|█████████████████████████████████████████| 447k/447k [00:01<00:00, 386kB/s]


Successfully loaded PROTEINS.


/usr/local/lib/python3.8/dist-packages/sklearn/preprocessing/_encoders.py:868: FutureWarning: `sparse` was renamed to `sparse_output` in version 1.2 and will be removed in 1.4. `sparse_output` is ignored unless you leave `sparse` to its default value.
  warnings.warn(


In [31]:
proteinas[0]

Graph(n_nodes=42, n_node_features=1, n_edge_features=None, n_labels=3)

In [32]:
split = int(0.8 * len(proteinas))
dataset_train, dataset_test = proteinas[:split], proteinas[split:]

In [33]:
proteinas.n_labels

3

In [11]:

batch_size = 1
loader_train = DisjointLoader(dataset_train, batch_size=batch_size, epochs=200, shuffle=False)
loader_test = DisjointLoader(dataset_test, batch_size=batch_size)


In [12]:
from spektral.models.gcn import GCN

model = GCN(n_labels=proteinas.n_labels)
optimizer = Adam(learning_rate=0.01)
loss_fn = CategoricalCrossentropy()

In [13]:
# Decorate the function with @tf.function to compile as a TensorFlow graph
# Use the input_signature from loader_train and relax shapes for varying graph sizes
@tf.function(input_signature=loader_train.tf_signature(), experimental_relax_shapes=True)
def train_step(inputs, target):
    print("target:",str(target))
    # Create a GradientTape context to record operations for automatic differentiation
    with tf.GradientTape() as tape:
        # Compute model predictions with the inputs, set training=True for training-specific behaviors
        predictions = model(inputs, training=True)
        print("pred:",str(predictions))
        #predictions = tf.argmax(predictions1,axis=1)
        # Calculate the loss using the provided loss_fn and add the model's regularization losses
        loss = loss_fn(target, predictions) + sum(model.losses)

    # Compute gradients of the loss with respect to the model's trainable variables
    gradients = tape.gradient(loss, model.trainable_variables)
    # Apply the gradients to the model's variables using the optimizer's apply_gradients method
    optimizer.apply_gradients(zip(gradients, model.trainable_variables))

    # Compute the accuracy using the categorical_accuracy function from TensorFlow
    # Calculate the mean accuracy using tf.reduce_mean
    acc = tf.reduce_mean(categorical_accuracy(target, predictions))

    # Return the loss and accuracy as output
    return loss, acc


In [14]:
def evaluate(loader):
    output = []
    step = 0
    while step < loader.steps_per_epoch:
        step += 1
        inputs, target = loader.__next__()
        pred = model(inputs, training=False)
        outs = (
            loss_fn(target, pred),
            tf.reduce_mean(categorical_accuracy(target, pred)),
            len(target),  # Keep track of batch size
        )
        output.append(outs)
        if step == loader.steps_per_epoch:
            output = np.array(output)
            return np.average(output[:, :-1], 0, weights=output[:, -1])


In [16]:
# Initialize the epoch and step counters to -1
# Create an empty list for storing training results
epoch = step = -1
results = []

# Iterate through the batches in the loader_train data loader
for batch in loader_train:
    # Increment the step counter
    step += 1

    # Execute the train_step function with the current batch
    # Obtain the loss and accuracy
    loss, acc = train_step(*batch)

    # Append the loss and accuracy to the results list
    results.append((loss, acc))

    # Check if the current step is equal to the number of steps per epoch (loader_train.steps_per_epoch)
    if step == loader_train.steps_per_epoch:
        # Reset the step counter to 0
        # Increment the epoch counter
        step = 0
        epoch += 1

        # Evaluate the model on the test set using the evaluate function (which should be defined beforehand)
        # Store the test results in results_te
        results_te = evaluate(loader_test) # CAMBIO A loader_val

        # Print the epoch number, mean training loss and accuracy, and test loss and accuracy
        print(
            "Ep. {} - Loss: {:.3f} - Acc: {:.3f} - Val loss: {:.3f} - Val acc: {:.3f}".format(
                epoch, *np.mean(results, 0), *results_te
            )
        )

        # Reset the results list to start collecting results for the next epoch
        results = []


ValueError: Shapes (1, 3) and (14, 3) are incompatible

In [17]:
algotest=loader_test.__next__()

In [18]:
algotest[0][0].shape,algotest[0][1].shape,algotest[0][2].shape,algotest[1].shape

((6, 1), TensorShape([6, 6]), (6,), (1, 3))

In [35]:
proteinas[0].y

array([1., 0., 0.])

In [20]:
algo=loader_train.__next__()

In [21]:
algo[0][0].shape, algo[0][1].shape, algo[0][2].shape, algo[1].shape

((10, 1), TensorShape([10, 10]), (10,), (1, 3))